<a href="https://colab.research.google.com/github/BlancaHoma14/tecnm-lenguajes-automatas-2/blob/main/laboratorios/sesion29/sesion29_laboratorio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lenguajes y Autómatas II (SCD-1016)
## Sesión 29 · Optimización: Eliminación de Código Muerto (Dead Code Elimination)

**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`) · **Repositorio maestro:** https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2

---

> **Pregunta guía:** ¿Cómo distingue un compilador el código que nunca se ejecutará del que sí importa?
>
> **Objetivo de la sesión:** Detectar instrucciones inalcanzables después de un GOTO o un RETURN y compactar la lista de cuádruplos conservando las etiquetas.

**Entregable:** tu función eliminar_muerto() en artesano_sesion29_c.c, el harness en verde y la bitácora en auditorias/auditoria_sesion29.md

**Tiempo:** 60 min · **Fases:** Artesano (manual) → Científico (IA con contrato) → Auditor (harness)

> ¿Dudas de C? Consulta la [guía de C](https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2/blob/main/docs/guia-c.md) antes de empezar.


## 1. ¿Qué vas a construir?

Recibes un arreglo con n instrucciones TAC y debes compactarlo en el mismo arreglo. Conserva todo hasta un GOTO o RETURN y descarta lo que sigue hasta la próxima etiqueta; devuelve cuántas instrucciones quedaron.

Al final de la sesión tendrás: **Una función en C que elimina instrucciones inalcanzables y devuelve el nuevo tamaño de la lista.**

Los conceptos y el porqué de esta sesión están en la presentación
([sesión 29](https://ijmarquezl.github.io/tecnm-lenguajes-automatas-2/presentaciones/sesion29.html)). Aquí solo está el trabajo: qué se
recibe, qué debe salir y cómo se comprueba.


## 2. El contrato exacto

El archivo `api_sesion29.h` declara la interfaz que el harness sabe probar.
Tu implementación **debe** respetar esas firmas; si las cambias, el harness no compila.

```c
#include "api_sesion29.h"

int eliminar_muerto(char lineas[][64], int n);
```

- Una etiqueta termina en dos puntos y siempre es alcanzable: corta la zona muerta.
- El GOTO y el RETURN se conservan; solo se descarta lo que viene después.
- Compacta en el mismo arreglo y no cambies el orden de las instrucciones que sobreviven.

> El harness compila con `-Wall -Wextra -Werror`: una advertencia del compilador cuenta como error.
>
> El harness compila `harness_sesion29_c.c` junto con `artesano_sesion29_c.c`
> (o `ia_sesion29_c.c`). Por eso tu archivo **no debe tener `main()`**: el `main()`
> es del harness.


## 3. Cómo se califica

### Casos que evalúa el harness

| # | Caso | Resultado esperado |
| :-: | :--- | :--- |
| 1 | `tras GOTO: nuevo tamano` | 4 |
| 2 | `tras GOTO: etiqueta viva` | `L1:` |
| 3 | `tras RETURN: nuevo tamano` | 4 |
| 4 | `tras RETURN: etiqueta viva` | `L5:` |
| 5 | `sin codigo muerto` | 3 |
| 6 | `lista vacia` | 0 |

### Salida literal que debes obtener

```text
=== AUDITORIA SESION 29 - Eliminacion de codigo inalcanzable ===
Caso                               esperado:              obtenido:              resultado
[ 1] tras GOTO: nuevo tamano            esperado: 4            obtenido: 4            OK
[ 2] tras GOTO: etiqueta viva           esperado: L1:          obtenido: L1:          OK
[ 3] tras RETURN: nuevo tamano          esperado: 4            obtenido: 4            OK
[ 4] tras RETURN: etiqueta viva         esperado: L5:          obtenido: L5:          OK
[ 5] sin codigo muerto                  esperado: 3            obtenido: 3            OK
[ 6] lista vacia                        esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion29.md
```

Los números entre corchetes son el contador de casos. Mientras algo falle verás
`FALLA` en esa línea, el total será menor y el programa terminará con código de
salida 1.

### Criterios de aceptación

- [ ] Toda instrucción posterior a un GOTO o RETURN se descarta hasta la próxima etiqueta.
- [ ] Las etiquetas y las instrucciones alcanzables conservan su orden original.
- [ ] Una lista vacía devuelve 0 sin leer memoria fuera del arreglo.


## Paso 0 · Entorno


In [1]:
import os

if os.system('which gcc > /dev/null 2>&1') != 0:
    print('⏳ Instalando compilador...')
    !apt-get update -qq && apt-get install -y -qq gcc > /dev/null 2>&1

print('✅ Entorno listo:')
!gcc --version | head -n 1


✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


## Fase 1 · El Artesano (15 min, sin IA)

Escribe a mano la implementación en `artesano_sesion29_c.c`. El archivo ya trae la
firma y un `TODO`: edita el contenido de la celda `%%writefile` que está abajo (o el
archivo en el panel **Archivos** de Colab) y vuelve a ejecutarla para guardar los
cambios. Hazlo sin consultar a la IA. Si te atoras, anota en tu bitácora exactamente
en qué te atoraste; eso también es evidencia.


In [2]:
%%writefile api_sesion29.h
/* api_sesion29.h - contrato de la Sesion 29
 * Eliminación de código inalcanzable
 *
 * Esta cabecera es el contrato: el harness y tu implementacion
 * deben incluirla. Si cambias una firma, el harness dejara de compilar.
 */
#ifndef API_SESION29_H
#define API_SESION29_H

int eliminar_muerto(char lineas[][64], int n);

#endif /* API_SESION29_H */


Writing api_sesion29.h


In [3]:
%%writefile artesano_sesion29_c.c

#include "api_sesion29.h"
#include <string.h>

static int es_etiqueta(const char *linea) {
    size_t i;
    size_t longitud;

    if (linea == NULL) {
        return 0;
    }

    longitud = strlen(linea);

    if (longitud == 0) {
        return 0;
    }

    for (i = longitud; i > 0; i--) {
        if (linea[i - 1] == ':') {
            return 1;
        }

        if (linea[i - 1] != ' ' &&
            linea[i - 1] != '\t' &&
            linea[i - 1] != '\n' &&
            linea[i - 1] != '\r') {
            return 0;
        }
    }

    return 0;
}

static int es_salto_o_retorno(const char *linea) {
    if (linea == NULL) {
        return 0;
    }

    if (strncmp(linea, "GOTO ", 5) == 0 ||
        strcmp(linea, "GOTO") == 0 ||
        strncmp(linea, "RETURN", 6) == 0) {
        return 1;
    }

    return 0;
}

int eliminar_muerto(char lineas[][64], int n) {
    int i;
    int escribir = 0;
    int muerto = 0;

    if (lineas == NULL || n <= 0) {
        return 0;
    }

    for (i = 0; i < n; i++) {

        if (muerto) {
            if (es_etiqueta(lineas[i])) {
                muerto = 0;
            } else {
                continue;
            }
        }

        if (escribir != i) {
            strcpy(lineas[escribir], lineas[i]);
        }

        escribir++;

        if (es_salto_o_retorno(lineas[i])) {
            muerto = 1;
        }
    }

    return escribir;
}



Writing artesano_sesion29_c.c


## Fase 2 · El Científico (10 min)

Redacta el contrato **C-R-E-O** (Contexto, Rol, Especificación, Output) para pedirle
la misma función a la IA, y pega el resultado en la celda de `ia_sesion29_c.c`. La
firma debe ser idéntica a la del contrato; si la IA la cambia, corrígela antes de
compilar.


**CONTRATO C-R-E-O**

**CONTEXTO:** Estoy desarrollando la Sesión 29 de Lenguajes y Autómatas II (SCD-1016). El tema es la eliminación de código muerto (Dead Code Elimination) en código intermedio TAC. Trabajo con C11 y GCC.

**ROL:** Actúa como ingeniero de software especializado en compiladores, optimización de código y programación en C.

**ESPECIFICACIÓN:** Implementa exactamente la siguiente función:
int eliminar_muerto(char lineas[][64], int n);
Requisitos:
1. Detectar instrucciones GOTO y RETURN.
2. Conservar la instrucción GOTO o RETURN.
3. Eliminar todas las instrucciones posteriores hasta encontrar una etiqueta.
4. Una etiqueta se reconoce porque termina en dos puntos.
5. Conservar las etiquetas porque siempre son alcanzables.
6. Después de una etiqueta, volver a conservar las instrucciones.
7. Compactar el arreglo original sin utilizar memoria dinámica.
8. Mantener el orden original de las instrucciones que sobreviven.
9. Devolver la nueva cantidad de instrucciones.
10. Si la lista está vacía, devolver 0.
11. Validar el apuntador del arreglo y el valor de n.
12. Reconocer correctamente las palabras GOTO y RETURN, sin confundirlas con identificadores que solamente comienzan con esas letras.
13. Considerar espacios y tabulaciones al identificar instrucciones.

**OUTPUT:** Entrega código modular en C11, sin función main(), respetando exactamente la firma original. Explica las mejoras respecto al código manual y analiza la complejidad temporal y espacial.

Debe compilar con:

gcc -std=c11 -Wall -Wextra -Werror

In [4]:
%%writefile ia_sesion29_c.c

#include "api_sesion29.h"
#include <string.h>
#include <ctype.h>

static int es_etiqueta_ia(const char *linea) {
    size_t longitud;

    if (linea == NULL) {
        return 0;
    }

    longitud = strlen(linea);

    while (longitud > 0 &&
           isspace((unsigned char)linea[longitud - 1])) {
        longitud--;
    }

    return longitud > 0 && linea[longitud - 1] == ':';
}

static int palabra_completa(const char *linea,
                            const char *palabra) {
    size_t longitud;

    while (*linea != '\0' &&
           isspace((unsigned char)*linea)) {
        linea++;
    }

    longitud = strlen(palabra);

    if (strncmp(linea, palabra, longitud) != 0) {
        return 0;
    }

    return linea[longitud] == '\0' ||
           isspace((unsigned char)linea[longitud]);
}

static int es_terminador_ia(const char *linea) {
    return palabra_completa(linea, "GOTO") ||
           palabra_completa(linea, "RETURN");
}

int eliminar_muerto(char lineas[][64], int n) {
    int i;
    int escribir = 0;
    int codigo_muerto = 0;

    if (lineas == NULL || n <= 0) {
        return 0;
    }

    for (i = 0; i < n; i++) {

        if (codigo_muerto) {
            if (es_etiqueta_ia(lineas[i])) {
                codigo_muerto = 0;
            } else {
                continue;
            }
        }

        if (escribir != i) {
            memcpy(lineas[escribir], lineas[i], 64);
        }

        escribir++;

        if (es_terminador_ia(lineas[i])) {
            codigo_muerto = 1;
        }
    }

    return escribir;
}



Writing ia_sesion29_c.c


## Fase 3 · El Auditor (10 min)

El harness de abajo **no aprueba nada por sí solo**: ejecuta tu función caso por caso,
la compara contra el valor esperado e imprime `OK` o `FALLA`. Léelo antes de ejecutarlo:
los casos te dicen qué se está midiendo.


In [5]:
%%writefile harness_sesion29_c.c
/* harness_sesion29_c.c - auditoria automatica de la Sesion 29
 *
 * Compilacion:
 *   gcc -std=c11 -Wall -Wextra -Werror harness_sesion29_c.c artesano_sesion29_c.c -o test_artesano
 *
 * El harness NO aprueba nada por si solo: cada linea se calcula
 * ejecutando tu implementacion y comparandola contra el valor esperado.
 * Si algun caso falla, el programa termina con codigo de salida 1.
 */
#include <stdio.h>
#include <string.h>
#include "api_sesion29.h"

static int probadas = 0;
static int fallidas = 0;

static void verificar_str(const char *caso, const char *esperado,
                          const char *obtenido) {
    int ok = (obtenido != NULL) && (strcmp(esperado, obtenido) == 0);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12s obtenido: %-12s %s\n",
           probadas, caso, esperado, obtenido ? obtenido : "(NULL)",
           ok ? "OK" : "FALLA");
}

static void verificar_int(const char *caso, long esperado, long obtenido) {
    int ok = (esperado == obtenido);
    probadas++;
    if (!ok) fallidas++;
    printf("[%2d] %-34s esperado: %-12ld obtenido: %-12ld %s\n",
           probadas, caso, esperado, obtenido, ok ? "OK" : "FALLA");
}

int main(void) {
    printf("=== AUDITORIA SESION 29 - Eliminacion de codigo inalcanzable ===\n");
    printf("Caso                               esperado:              obtenido:              resultado\n");

    { char l[8][64] = {"_t0 = 1", "GOTO L1", "_t1 = 2", "L1:", "_t2 = 3"}; int m = eliminar_muerto(l, 5); verificar_int("tras GOTO: nuevo tamano", (long)(4), (long)(m)); }
    { char l[8][64] = {"_t0 = 1", "GOTO L1", "_t1 = 2", "L1:", "_t2 = 3"}; eliminar_muerto(l, 5); verificar_str("tras GOTO: etiqueta viva", "L1:", l[2]); }
    { char l[8][64] = {"_t0 = 1", "RETURN", "_t1 = 2", "L5:", "_t2 = 3"}; int m = eliminar_muerto(l, 5); verificar_int("tras RETURN: nuevo tamano", (long)(4), (long)(m)); }
    { char l[8][64] = {"_t0 = 1", "RETURN", "_t1 = 2", "L5:", "_t2 = 3"}; eliminar_muerto(l, 5); verificar_str("tras RETURN: etiqueta viva", "L5:", l[2]); }
    { char l[8][64] = {"_t0 = 1", "_t1 = 2", "_t2 = _t0 + _t1"}; int m = eliminar_muerto(l, 3); verificar_int("sin codigo muerto", (long)(3), (long)(m)); }
    { char l[4][64] = {{0}}; int m = eliminar_muerto(l, 0); verificar_int("lista vacia", (long)(0), (long)(m)); }

    printf("\nRESULTADO: %d/%d pruebas superadas\n",
           probadas - fallidas, probadas);
    if (fallidas != 0) {
        printf("Aun hay %d caso(s) sin resolver. Revisa tu implementacion.\n",
               fallidas);
        return 1;
    }
    printf("TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion29.md" "\n");
    return 0;
}


Writing harness_sesion29_c.c


In [6]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion29_c.c artesano_sesion29_c.c -o test_artesano && ./test_artesano

=== AUDITORIA SESION 29 - Eliminacion de codigo inalcanzable ===
Caso                               esperado:              obtenido:              resultado
[ 1] tras GOTO: nuevo tamano            esperado: 4            obtenido: 4            OK
[ 2] tras GOTO: etiqueta viva           esperado: L1:          obtenido: L1:          OK
[ 3] tras RETURN: nuevo tamano          esperado: 4            obtenido: 4            OK
[ 4] tras RETURN: etiqueta viva         esperado: L5:          obtenido: L5:          OK
[ 5] sin codigo muerto                  esperado: 3            obtenido: 3            OK
[ 6] lista vacia                        esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion29.md


In [7]:
!gcc -std=c11 -Wall -Wextra -Werror harness_sesion29_c.c ia_sesion29_c.c -o test_ia && ./test_ia

=== AUDITORIA SESION 29 - Eliminacion de codigo inalcanzable ===
Caso                               esperado:              obtenido:              resultado
[ 1] tras GOTO: nuevo tamano            esperado: 4            obtenido: 4            OK
[ 2] tras GOTO: etiqueta viva           esperado: L1:          obtenido: L1:          OK
[ 3] tras RETURN: nuevo tamano          esperado: 4            obtenido: 4            OK
[ 4] tras RETURN: etiqueta viva         esperado: L5:          obtenido: L5:          OK
[ 5] sin codigo muerto                  esperado: 3            obtenido: 3            OK
[ 6] lista vacia                        esperado: 0            obtenido: 0            OK

RESULTADO: 6/6 pruebas superadas
TODO EN VERDE. Documenta tu auditoria en auditorias/auditoria_sesion29.md


## Resultado esperado

Si tu implementación es correcta, la última línea del harness será:

```text
RESULTADO: 6/6 pruebas superadas
```

y la celda terminará sin error (código de salida 0). Compara tu salida completa
contra la de la sección 3: debe ser idéntica, línea por línea.


## Entrega

1. Copia `plantillas/auditoria_sesion29.md` a `auditorias/auditoria_sesion29.md` en tu fork.
2. Adjunta la salida del harness (las 6 líneas con `OK`).
3. Explica en el reporte crítico qué caso borde omitió la IA y cómo lo detectaste.
4. Haz commit y sincroniza tu fork.

Este laboratorio alimenta el 40% de laboratorios y es la base del Hito correspondiente
del Proyecto Integrador.
